# Phase 4 (part 1) — Dependency Graph Validation

The archaeologist's signature move: reason over **call** and **inherit** edges between symbols to answer questions retrieval alone can't.

**Run first:** `uv run python -m archaeologist.indexing.graph`

Resolution is name-based (Python is dynamic), so edges are approximate — builtins and overly-ambiguous names are dropped. Good enough for impact, coupling, and path questions.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from sqlalchemy import func, select
from archaeologist.models.db import session_scope
from archaeologist.models.entities import Repo, SymbolEdge
from archaeologist.retrieval.graph_queries import (
    call_path, find_symbol, most_coupled_files, who_depends_on,
)
print("repo root:", root)

## Edge counts

In [ ]:
with session_scope() as s:
    total = s.scalar(select(func.count()).select_from(SymbolEdge))
    print(f"total edges: {total}")
    for et, n in s.execute(select(SymbolEdge.edge_type, func.count()).group_by(SymbolEdge.edge_type)):
        print(f"  {et:8}: {n}")

## "What breaks if I remove X?"
Reverse dependencies — every symbol that calls or inherits the target.

In [ ]:
with session_scope() as s:
    for target in ["Flask.dispatch_request", "View", "Flask.preprocess_request"]:
        sym = find_symbol(s, target)
        print(f"\n# remove {target} ->")
        if not sym:
            print("  (not found)"); continue
        deps = who_depends_on(s, sym.id)
        if not deps:
            print("  nothing internal depends on it")
        for et, dep in deps[:10]:
            print(f"  [{et:7}] {dep.qualified_name:34.34} {dep.file_path}:{dep.start_line}")

## Which modules are most tightly coupled?
Test files call everything, so the architectural signal is clearer with them excluded.

In [ ]:
with session_scope() as s:
    repo = s.scalar(select(Repo))
    print("all files:")
    for a, b, n in most_coupled_files(s, repo.id, limit=5):
        print(f"  {n:3}  {a} -> {b}")
    print("\ntests excluded (real architecture):")
    for a, b, n in most_coupled_files(s, repo.id, limit=8, exclude_tests=True):
        print(f"  {n:3}  {a} -> {b}")

## Execution-path sketch
BFS over outgoing call edges from `Flask.full_dispatch_request` — Flask's request lifecycle, reconstructed.

In [ ]:
with session_scope() as s:
    start = find_symbol(s, "Flask.full_dispatch_request")
    if start:
        for depth, sym in call_path(s, start.id, max_depth=3):
            print(f"  {'  ' * depth}d{depth} {sym.qualified_name}  ({sym.file_path}:{sym.start_line})")

## Summary

In [ ]:
with session_scope() as s:
    total = s.scalar(select(func.count()).select_from(SymbolEdge))
print("Phase 4 (graph) —", "OK ✅" if total > 0 else "NO EDGES ❌")
print(f"  dependency edges: {total}")